In [1]:
import pandas as pd
from pathlib import Path

In [2]:
RESULTS_DIR = Path("tournament_results_probe")  # change if needed

dfs = []
for f in RESULTS_DIR.glob("*.csv"):
    df = pd.read_csv(f)
    df["source_file"] = f.name
    dfs.append(df)

combined = pd.concat(dfs, ignore_index=True)

print(f"Loaded {len(dfs)} files, {len(combined)} total games")
combined.head()

Loaded 6 files, 60 total games


,opening_idx,opening_fen,white_policy,black_policy,discrete,result,termination,total_plies,white_budget_remaining,black_budget_remaining,white_nodes_total,black_nodes_total,source_file
0,1,rnb1k2r/pp2qppp/3bp3/3n4/2BP4/5N2/PP1B1PPP/RN1...,TokenBucket,FixedUniform,False,1/2-1/2,draw_rule,40,9499662,6597129,500338,3402871,TokenBucket_vs_FixedUniform_continuous.csv
1,1,rnb1k2r/pp2qppp/3bp3/3n4/2BP4/5N2/PP1B1PPP/RN1...,FixedUniform,TokenBucket,False,1-0,resign_adjudication,87,1194164,1019139,8805836,8980861,TokenBucket_vs_FixedUniform_continuous.csv
2,0,r1bqk2r/1ppn1ppp/3p1n2/p1b1p3/P1B1P3/2P2N1P/1P...,TokenBucket,FixedUniform,False,1-0,budget_forfeit,101,959975,0,9040025,10000096,TokenBucket_vs_FixedUniform_continuous.csv
3,0,r1bqk2r/1ppn1ppp/3p1n2/p1b1p3/P1B1P3/2P2N1P/1P...,FixedUniform,TokenBucket,False,1-0,resign_adjudication,95,391706,1367723,9608294,8632277,TokenBucket_vs_FixedUniform_continuous.csv
4,2,r2qk2r/p2bppbp/n1p2np1/1p1p4/2P5/3P1NP1/PPQ1PP...,TokenBucket,FixedUniform,False,0-1,resign_adjudication,51,6272044,4996119,3727956,5003881,TokenBucket_vs_FixedUniform_continuous.csv


In [3]:
combined.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 60 entries, 0 to 59
Data columns (total 13 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   opening_idx             60 non-null     int64 
 1   opening_fen             60 non-null     object
 2   white_policy            60 non-null     object
 3   black_policy            60 non-null     object
 4   discrete                60 non-null     bool  
 5   result                  60 non-null     object
 6   termination             60 non-null     object
 7   total_plies             60 non-null     int64 
 8   white_budget_remaining  60 non-null     int64 
 9   black_budget_remaining  60 non-null     int64 
 10  white_nodes_total       60 non-null     int64 
 11  black_nodes_total       60 non-null     int64 
 12  source_file             60 non-null     object
dtypes: bool(1), int64(6), object(6)
memory usage: 5.8+ KB


In [5]:
combined.describe()

,opening_idx,total_plies,white_budget_remaining,black_budget_remaining,white_nodes_total,black_nodes_total
count,60.000000,60.000000,6.000000e+01,6.000000e+01,6.000000e+01,6.000000e+01
mean,2.000000,86.800000,4.753490e+06,4.609370e+06,5.246510e+06,5.390632e+06
std,1.426148,37.155103,3.382285e+06,3.250973e+06,3.382285e+06,3.250975e+06
min,0.000000,28.000000,0.000000e+00,0.000000e+00,5.003380e+05,3.502470e+05
25%,1.000000,62.000000,1.135617e+06,1.324167e+06,1.694990e+06,2.645312e+06
50%,2.000000,89.000000,5.496437e+06,4.493066e+06,4.503563e+06,5.506934e+06
75%,3.000000,106.000000,8.305010e+06,7.354688e+06,8.864383e+06,8.675833e+06
max,4.000000,194.000000,9.499662e+06,9.649753e+06,1.000001e+07,1.000010e+07


In [6]:
combined["result"].value_counts()

result
1-0        23
1/2-1/2    20
0-1        17
Name: count, dtype: int64

In [7]:
combined["termination"].value_counts()

termination
resign_adjudication    36
draw_adjudication      14
draw_rule               6
budget_forfeit          2
checkmate               2
Name: count, dtype: int64

In [9]:
def tokenbucket_score(df):
    wins = draws = losses = 0
    
    for _, row in df.iterrows():
        if row["result"] == "1/2-1/2":
            draws += 1
        elif row["result"] == "1-0":
            if row["white_policy"] == "TokenBucket":
                wins += 1
            elif row["black_policy"] == "TokenBucket":
                losses += 1
        elif row["result"] == "0-1":
            if row["black_policy"] == "TokenBucket":
                wins += 1
            elif row["white_policy"] == "TokenBucket":
                losses += 1
    
    total = len(df)
    score = wins + 0.5 * draws
    return wins, draws, losses, 100 * score / total

tokenbucket_score(combined) # WDL + %

(3, 20, 37, 21.666666666666668)

In [11]:
combined.groupby("discrete").size()

for mode, df in combined.groupby("discrete"):
    print("\nDiscrete =", mode)
    print(tokenbucket_score(df))


Discrete = False
(3, 12, 15, 30.0)

Discrete = True
(0, 8, 22, 13.333333333333334)


In [12]:
combined["termination"].value_counts(normalize=True)

termination
resign_adjudication    0.600000
draw_adjudication      0.233333
draw_rule              0.100000
budget_forfeit         0.033333
checkmate              0.033333
Name: proportion, dtype: float64

In [13]:
combined[combined["termination"] == "budget_forfeit"].head()

,opening_idx,opening_fen,white_policy,black_policy,discrete,result,termination,total_plies,white_budget_remaining,black_budget_remaining,white_nodes_total,black_nodes_total,source_file
2,0,r1bqk2r/1ppn1ppp/3p1n2/p1b1p3/P1B1P3/2P2N1P/1P...,TokenBucket,FixedUniform,False,1-0,budget_forfeit,101,959975,0,9040025,10000096,TokenBucket_vs_FixedUniform_continuous.csv
9,3,rnbqk2r/ppp2ppp/5n2/2b5/2N1p3/3P4/PPP2PPP/RNBQ...,FixedUniform,TokenBucket,False,0-1,budget_forfeit,100,0,8749029,10000009,1250971,TokenBucket_vs_FixedUniform_continuous.csv


In [14]:
combined[["white_nodes_total", "black_nodes_total"]].describe()

,white_nodes_total,black_nodes_total
count,6.000000e+01,6.000000e+01
mean,5.246510e+06,5.390632e+06
std,3.382285e+06,3.250975e+06
min,5.003380e+05,3.502470e+05
25%,1.694990e+06,2.645312e+06
50%,4.503563e+06,5.506934e+06
75%,8.864383e+06,8.675833e+06
max,1.000001e+07,1.000010e+07


In [15]:
def extract_nodes(row):
    if row["white_policy"] == "TokenBucket":
        return row["white_nodes_total"], row["black_nodes_total"]
    else:
        return row["black_nodes_total"], row["white_nodes_total"]

tmp = combined.apply(lambda r: extract_nodes(r), axis=1, result_type="expand")
tmp.columns = ["tb_nodes", "opp_nodes"]

tmp.describe()

,tb_nodes,opp_nodes
count,6.000000e+01,6.000000e+01
mean,3.970770e+06,6.666372e+06
std,3.488537e+06,2.480856e+06
min,3.502470e+05,1.400919e+06
25%,1.100822e+06,4.752705e+06
50%,2.413686e+06,6.854486e+06
75%,8.234288e+06,9.145836e+06
max,9.970808e+06,1.000010e+07


In [16]:
def classify(row):
    if row["result"] == "1/2-1/2":
        return "draw"
    if row["result"] == "1-0":
        return "win" if row["white_policy"] == "TokenBucket" else "loss"
    if row["result"] == "0-1":
        return "win" if row["black_policy"] == "TokenBucket" else "loss"

combined["tb_result"] = combined.apply(classify, axis=1)

combined["tb_result"].value_counts()
pd.crosstab(combined["tb_result"], combined["termination"])

tb_result
loss    37
draw    20
win      3
Name: count, dtype: int64

In [18]:
for name, df in combined.groupby("source_file"):
    print("\n", name)
    print(tokenbucket_score(df))


 TokenBucket_vs_FixedUniform_continuous.csv
(3, 3, 4, 45.0)

 TokenBucket_vs_FixedUniform_discrete.csv
(0, 2, 8, 10.0)

 TokenBucket_vs_Hyatt_continuous.csv
(0, 3, 7, 15.0)

 TokenBucket_vs_Hyatt_discrete.csv
(0, 4, 6, 20.0)

 TokenBucket_vs_SolakVuckovic_continuous.csv
(0, 6, 4, 30.0)

 TokenBucket_vs_SolakVuckovic_discrete.csv
(0, 2, 8, 10.0)


In [19]:
pd.crosstab(combined["tb_result"], combined["termination"])

termination,budget_forfeit,checkmate,draw_adjudication,draw_rule,resign_adjudication
tb_result,,,,,
draw,0,0,14,6,0
loss,0,2,0,0,35
win,2,0,0,0,1


In [20]:
tmp.groupby(combined["tb_result"]).mean()

,tb_nodes,opp_nodes
tb_result,,
draw,1.871122e+06,8.015008e+06
loss,5.124518e+06,5.742624e+06
win,3.738852e+06,9.068369e+06


In [21]:
combined.groupby(["discrete", "tb_result"]).size()

discrete  tb_result
False     draw         12
          loss         15
          win           3
True      draw          8
          loss         22
dtype: int64